In [1]:
import sys
sys.path.append('../../../TaskExecutionTimeMining/')

import os
import shutil
import pickle
import numpy as np
from pathlib import Path
import subprocess

from divide_and_conquer_ib import *

In [2]:
model_path = "../../../../models/advanced/bpic_2017/concept-name_resource"
event_log_path = "../../transformed_event_logs/BPIC_2017_all_train.pickle"
screen_prefix = "BPIC17_cr_"


target_column = 'duration_seconds'
continuous_columns = [
    #'seconds_in_day',
    #'case:RequestedAmount_start'
]
categorical_columns = [
    'concept:name',
    'org:resource_start',
    #'day_of_week'
]


In [3]:
with open(event_log_path, "rb") as f:
    event_log = pickle.load(f)

transformed_event_log = event_log.copy()

transformations = dict()
for num_attr in continuous_columns + [target_column]:
    transformed_event_log[num_attr] = np.log1p(transformed_event_log[num_attr]+1)
    m = transformed_event_log[num_attr].mean()
    std = transformed_event_log[num_attr].std()
    transformed_event_log[num_attr] = (transformed_event_log[num_attr] - m) / std
    transformations[num_attr] = (m, std)

/tmp/ipykernel_10189/25019715.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  event_log = pickle.load(f)


In [4]:
# clear the model directory
ignore_file = "drbart_variable.r"

for entry in os.listdir(model_path):
    if entry == ignore_file:
        continue  # Skip this file
    path = os.path.join(model_path, entry)
    if os.path.isfile(path) or os.path.islink(path):
        os.unlink(path)  # Remove file or symlink
    elif os.path.isdir(path):
        shutil.rmtree(path)  # Remove directory and all contents


In [5]:
res = divide_and_conquer_ib(
    transformed_event_log,
    target_column=target_column,
    continuous_columns=continuous_columns,
    categorical_columns=categorical_columns,
    n_clusters=32
)

Clustering concept:name


/home/LordKunkler/.local/share/virtualenvs/TaskExecutionTimeMining-yRnjZRF7/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/LordKunkler/.local/share/virtualenvs/TaskExecutionTimeMining-yRnjZRF7/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(


X discrete (categorical), Adjusted n_bins_x: 41, Max X_d index: 40, Unique X_d bins: 41
Y continuous (quantile bins), Adjusted n_bins_y: 435, Max Y_d index: 434, Unique Y_d bins: 435
Initial (Clusters: 41): Mutual Information I(T; Y) = 1.579209


Merging clusters:   0%|          | 0/9 [00:00<?, ?it/s]

Iteration 1 pairs:   0%|          | 0/820 [00:00<?, ?it/s]

...

Iteration 8 (Clusters: 33): Mutual Information I(T; Y) = 1.579069


Iteration 9 pairs:   0%|          | 0/528 [00:00<?, ?it/s]

Iteration 9 (Clusters: 32): Mutual Information I(T; Y) = 1.579031
New best MI: 1.5790308635054968 for column concept:name
Clustering org:resource_start


/home/LordKunkler/.local/share/virtualenvs/TaskExecutionTimeMining-yRnjZRF7/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/LordKunkler/.local/share/virtualenvs/TaskExecutionTimeMining-yRnjZRF7/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(


X discrete (categorical), Adjusted n_bins_x: 148, Max X_d index: 147, Unique X_d bins: 148
Y continuous (quantile bins), Adjusted n_bins_y: 435, Max Y_d index: 434, Unique Y_d bins: 435
Initial (Clusters: 148): Mutual Information I(T; Y) = 0.424528


In [6]:
# General version
unique_ids = np.unique(res[1])

# Create a dictionary of DataFrames
divided_event_logs = {uid: transformed_event_log[res[1] == uid].reset_index(drop=True) for uid in unique_ids}

In [7]:
# Assume routed_dfs is your dictionary of DataFrames
base_dir = Path(model_path)  # or any base directory name you like
base_dir.mkdir(exist_ok=True)

with open(base_dir / "gate.pickle", "wb") as f:
    pickle.dump(res, f)

with open(base_dir / "transformations.pickle", "wb") as f:
    pickle.dump(transformations, f)

for uid, sub_event_log in divided_event_logs.items():
    folder = base_dir / str(uid)
    folder.mkdir(exist_ok=True)
    sub_event_log.to_csv(folder / "data.csv", index=False)
    shutil.copy(
        model_path + "/drbart_variable.r",
        folder / "drbart_variable.r"
    )
    cmd = f"screen -dmS {screen_prefix+str(uid)} bash -c 'cd \"{folder}\" && ../../../../../src/DRBartModelTrainer/train_dr_bart.sh ../../../../../src/TaskExecutionTimeMining/drbart_training/numerical_stable_model/drbart_static.r > out.log 2>&1'"
    print(cmd)
    subprocess.run(cmd, shell=True)
    print(uid, sub_event_log.shape)

screen -dmS BPIC17_cr_0 bash -c 'cd "../../../../models/advanced/bpic_2017/concept-name_resource/0" && ../../../../../src/DRBartModelTrainer/train_dr_bart.sh ../../../../../src/TaskExecutionTimeMining/drbart_training/numerical_stable_model/drbart_static.r > out.log 2>&1'
0 (31, 233)
screen -dmS BPIC17_cr_1 bash -c 'cd "../../../../models/advanced/bpic_2017/concept-name_resource/1" && ../../../../../src/DRBartModelTrainer/train_dr_bart.sh ../../../../../src/TaskExecutionTimeMining/drbart_training/numerical_stable_model/drbart_static.r > out.log 2>&1'
1 (513, 233)
screen -dmS BPIC17_cr_2 bash -c 'cd "../../../../models/advanced/bpic_2017/concept-name_resource/2" && ../../../../../src/DRBartModelTrainer/train_dr_bart.sh ../../../../../src/TaskExecutionTimeMining/drbart_training/numerical_stable_model/drbart_static.r > out.log 2>&1'
2 (209, 233)
screen -dmS BPIC17_cr_3 bash -c 'cd "../../../../models/advanced/bpic_2017/concept-name_resource/3" && ../../../../../src/DRBartModelTrainer/train